In [ ]:
import json
import pandas as pd
from dotenv import load_dotenv
import os
from pathlib import Path

load_dotenv()


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "self-instruct").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Cannot find the project root")


project_root = find_project_root()
recent_prompts_path = project_root / "self-instruct" / "data" / "self_instructed_models_step_6.jsonl"
old_prompts_path = project_root / "self-instruct" / "data" / "self_instructed_legacy_models_step_6.jsonl"
corpus_path = project_root / "data" / "model_indices" / "e1_e2.json"
recent_prompts = []
old_prompts = []
corpus = []
with open(recent_prompts_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        recent_prompts.append(entry)
with open(old_prompts_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        old_prompts.append(entry)
with open(corpus_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        corpus.append(entry)

In [ ]:
df_old = pd.DataFrame(old_prompts)
df_old["domain"].unique()

In [ ]:
# manual mapping of domains of df to the unique domains in the corpus
mapping = {
    "reinforcement-learning": "Reinforcement Learning",
    "tabular-classification": "Tabular Tabular Classification",
    "tabular-regression": "Tabular Tabular Regression",
    "text-to-3d": "Computer Vision Text-to-3D",
    "image-to-3d": "Computer Vision Image-to-3D",
    "text-ranking": "Natural Language Processing Text Ranking",
    "sentence-similarity": "Natural Language Processing Sentence Similarity",
    "zero-shot-classification": "Natural Language Processing Zero-Shot Classification",
    "translation": "Natural Language Processing Translation",
    "summarization": "Natural Language Processing Summarization",
    "text-generation": "Natural Language Processing Text Generation",
    "feature-extraction": "Natural Language Processing Feature Extraction",
    "question-answering": "Natural Language Processing Question Answering",
    "table-question-answering": "Natural Language Processing Table Question Answering",
    "text-to-image": "Computer Vision Text-to-Image",
    "text-to-video": "Computer Vision Text-to-Video",
    "image-to-text": "Computer Vision Image-to-Text",
    "image-to-image": "Computer Vision Image-to-Image",
    "image-to-video": "Computer Vision Image-to-Video",
    "image-classification": "Computer Vision Image Classification",
    "image-segmentation": "Computer Vision Image Segmentation",
    "image-feature-extraction": "Computer Vision Image Feature Extraction",
    "object-detection": "Computer Vision Object Detection",
    "zero-shot-object-detection": "Computer Vision Zero-Shot Object Detection",
    "keypoint-detection": "Computer Vision Keypoint Detection",
    "mask-generation": "Computer Vision Mask Generation",
    "depth-estimation": "Computer Vision Depth Estimation",
    "unconditional-image-generation": "Computer Vision Unconditional Image Generation",
    "zero-shot-image-classification": "Computer Vision Zero-Shot Image Classification",
    "video-classification": "Computer Vision Video Classification",
    "video-to-video": "Computer Vision Video-to-Video",
    "video-text-to-text": "Multimodal Video-Text-to-Text",
    "visual-question-answering": "Multimodal Visual Question Answering",
    "document-question-answering": "Multimodal Document Question Answering",
    "visual-document-retrieval": "Multimodal Visual Document Retrieval",
    "image-text-to-text": "Multimodal Image-Text-to-Text",
    "image-text-to-image": "Multimodal Image-Text-to-Image",
    "image-text-to-video": "Multimodal Image-Text-to-Video",
    "audio-text-to-text": "Audio Audio Text-to-Text",
    "text-to-speech": "Audio Text-to-Speech",
    "text-to-audio": "Audio Text-to-Audio",
    "automatic-speech-recognition": "Audio Automatic Speech Recognition",
    "audio-to-audio": "Audio Audio-to-Audio",
    "audio-classification": "Audio Audio Classification",
    "any-to-any": "Multimodal Any-to-Any",
    "token-classification": "Natural Language Processing Token Classification",
    "text-classification": "Natural Language Processing Text Classification",
    "fill-mask": "Natural Language Processing Fill-Mask",
}


In [ ]:
df_new = pd.DataFrame(recent_prompts)
df_old = pd.DataFrame(old_prompts)
# change columns names to have domain instead of task
df_new.rename(
    columns={
        "Instruction": "instruction",
        "model_id": "model_name",
        "modelcard": "description",
    },
    inplace=True,
)
df_old.rename(
    columns={
        "Instruction": "instruction",
        "model_id": "model_name",
        "modelcard": "description",
    },
    inplace=True,
)

df_new["domain"] = df_new["domain"].map(mapping)

# for the old prompts, we need to search the model_id in the corpus to get the domain
model_id_to_domain = {}
for entry in corpus:
    model_id_to_domain[entry["model_name"]] = (
        entry["domain"] if "domain" in entry else None
    )
df_old["domain"] = df_old["model_name"].map(model_id_to_domain)

In [ ]:
df_old

In [ ]:
# print number of missing domains in df_old
missing_domains = df_old["domain"].isnull().sum()
print(f"Number of missing domains in old prompts: {missing_domains}")

# Gestisci created_at


In [ ]:
import numpy as np
import pandas as pd


def parse_created_at(s):
    s = s.copy()

    num = pd.to_numeric(s, errors="coerce")

    dt = pd.to_datetime(s, errors="coerce", utc=True)

    ms = num.notna() & (num > 1e12)
    sec = num.notna() & ~ms

    dt.loc[ms] = pd.to_datetime(num[ms], unit="ms", utc=True)
    dt.loc[sec] = pd.to_datetime(num[sec], unit="s", utc=True)

    return dt


df_new["created_at"] = parse_created_at(df_new["created_at"])
df_old["created_at"] = parse_created_at(df_old["created_at"])

In [ ]:
# split df_new in two dataframes according to the median created_at date which is in format YYYY-MM-DD
df_new["created_at"] = pd.to_datetime(df_new["created_at"])
median_date = df_new["created_at"].median()
# split
df_recent = df_new[df_new["created_at"] >= median_date]
df_old_recent = df_new[df_new["created_at"] < median_date]
# print min and max created_at of df_old_recent
print(df_old_recent["created_at"].min())
print(df_old_recent["created_at"].max())
print(df_recent["created_at"].min())
print(df_recent["created_at"].max())

# print sizes
print(f"Number of recent models: {len(df_recent)}")
print(f"Number of old recent models: {len(df_old_recent)}")
# check number of unique model names between df_old and df_old_recent
unique_old_models = set(df_old["model_name"].unique())
unique_old_recent_models = set(df_old_recent["model_name"].unique())
common_models = unique_old_models.intersection(unique_old_recent_models)
print(f"Number of common models between old and old recent: {len(common_models)}")
# nunique models in df_old_recent and df_recent
print(f"Number of unique models in old recent: {df_old_recent['model_name'].nunique()}")
print(f"Number of unique models in recent: {df_recent['model_name'].nunique()}")


# pick models that have at least 10 prompts, then sample 10 prompts for each model from df_old_recent and put to df_recent
eligible_models = (
    df_old_recent.groupby("model_name").size().loc[lambda counts: counts >= 10].index
)
sample_size = min(60, len(eligible_models))
if sample_size == 0:
    raise ValueError("No models in df_old_recent have at least 10 prompts")
selected_models = pd.Series(eligible_models).sample(sample_size, random_state=42)
prompts_to_add = []
for model in selected_models:
    model_prompts = df_old_recent[df_old_recent["model_name"] == model]
    sampled_prompts = model_prompts.sample(10, random_state=42)
    prompts_to_add.append(sampled_prompts)
    # remove sampled prompts from df_old_recent
    df_old_recent = df_old_recent.drop(sampled_prompts.index)
prompts_to_add_df = pd.concat(prompts_to_add)
# concatenate to df_recent
df_recent = pd.concat([df_recent, prompts_to_add_df])
# print sizes and stats as before
print("########################################################################")

print(f"New number of recent prompts: {len(df_recent)}")
print(f"New number of unique models in recent: {df_recent['model_name'].nunique()}")
print(f"New number of old recent models: {len(df_old_recent)}")
print(
    f"New number of unique models in old recent: {df_old_recent['model_name'].nunique()}"
)

In [ ]:
# put half of models of df_old in df_recent and the other half in df_old_recent
num_models_old = df_old["model_name"].nunique()
half_num_models_old = 70  # num_models_old // 2
old_models_list = df_old["model_name"].unique().tolist()
# pick randomly half of the models
import random

models_for_old_recent = random.sample(old_models_list, half_num_models_old)
models_for_recent = [
    model for model in old_models_list if model not in models_for_old_recent
]

df_old_recent_additional = df_old[df_old["model_name"].isin(models_for_old_recent)]
df_recent_additional = df_old[df_old["model_name"].isin(models_for_recent)]
# concatenate dataframes
experience_3 = pd.concat([df_old_recent, df_old_recent_additional], ignore_index=True)
experience_4 = pd.concat([df_recent, df_recent_additional], ignore_index=True)

# shuffle dataframes
experience_3 = experience_3.sample(frac=1, random_state=42).reset_index()
experience_4 = experience_4.sample(frac=1, random_state=42).reset_index()

In [ ]:
# drop models that compare less than 5 prompts in experience_3 and experience_4
min_prompts = 10
model_prompt_counts_3 = experience_3["model_name"].value_counts()
models_to_keep_3 = model_prompt_counts_3[model_prompt_counts_3 >= min_prompts].index
experience_3 = experience_3[experience_3["model_name"].isin(models_to_keep_3)]
model_prompt_counts_4 = experience_4["model_name"].value_counts()
models_to_keep_4 = model_prompt_counts_4[model_prompt_counts_4 >= min_prompts].index
experience_4 = experience_4[experience_4["model_name"].isin(models_to_keep_4)]
# print final sizes
print("########################################################################")
print(f"Final number of prompts in experience 3: {len(experience_3)}")
print(
    f"Final number of unique models in experience 3: {experience_3['model_name'].nunique()}"
)
print(f"Final number of prompts in experience 4: {len(experience_4)}")
print(
    f"Final number of unique models in experience 4: {experience_4['model_name'].nunique()}"
)


In [ ]:
import pandas as pd


# statistics
def print_stats(df, name, df_old):
    print(f"Statistics for {name}:")
    print(f"Number of prompts: {len(df)}")
    print(f"Number of unique models: {df['model_name'].nunique()}")
    print(f"Number of unique domains: {df['domain'].nunique()}")

    # created_at statistics
    if "created_at" in df.columns:
        # Normalize all timestamps to UTC to avoid tz-aware / tz-naive mix
        created_at_series = pd.to_datetime(df["created_at"], errors="coerce", utc=True)

        max_date = created_at_series.max()
        min_date = created_at_series.min()

        if pd.notna(max_date):
            print(f"Last date in created_at: {max_date}")
            print(f"First date in created_at: {min_date}")
        else:
            print("Last date in created_at: N/A")
    else:
        print("Column 'created_at' not found")

    # models also present in old prompts
    common_models = set(df["model_name"].unique()).intersection(
        set(df_old["model_name"].unique())
    )
    print(f"Number of models also in old prompts: {len(common_models)}")

    # minimum number of prompts per model
    model_prompt_counts = df["model_name"].value_counts()
    min_prompts = model_prompt_counts.min()
    print(f"Minimum number of prompts per model: {min_prompts}")
    print()


# calls
print_stats(experience_3, "Experience 3", df_old)

print_stats(
    experience_4, "Experience 4", pd.concat([df_old, df_old_recent], ignore_index=True)
)


In [ ]:
e3 = experience_3.groupby("domain").size().reset_index(name="counts")
e4 = experience_4.groupby("domain").size().reset_index(name="counts")
# join e3 and e4 on domain mantaining all domains
e3_e4 = pd.merge(e3, e4, on="domain", how="outer", suffixes=("_e3", "_e4"))
# convert to latex table transform all to integer without scientific notation
e3_e4["counts_e3"] = e3_e4["counts_e3"].fillna(0).astype(int)
e3_e4["counts_e4"] = e3_e4["counts_e4"].fillna(0).astype(int)
latex_table = e3_e4.to_latex(index=False)
print(latex_table)
e3_e4

In [ ]:
# split in train valdation and test (80/10/10) in stratified way according to model_name using sklearn
from sklearn.model_selection import train_test_split

train_3, temp_3 = train_test_split(
    experience_3, test_size=0.2, stratify=experience_3["model_name"], random_state=42
)
val_3, test_3 = train_test_split(
    temp_3, test_size=0.5, stratify=temp_3["model_name"], random_state=42
)
train_4, temp_4 = train_test_split(
    experience_4, test_size=0.2, stratify=experience_4["model_name"], random_state=42
)
val_4, test_4 = train_test_split(
    temp_4, test_size=0.5, stratify=temp_4["model_name"], random_state=42
)
# save to jsonl files
output_dir_exp3 = project_root / "data" / "raw" / "exp3"
output_dir_exp4 = project_root / "data" / "raw" / "exp4"
output_dir_exp3.mkdir(parents=True, exist_ok=True)
output_dir_exp4.mkdir(parents=True, exist_ok=True)
train_3.to_json(
    output_dir_exp3 / "exp3-train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
val_3.to_json(
    output_dir_exp3 / "exp3-val.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
test_3.to_json(
    output_dir_exp3 / "exp3-eval.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
train_4.to_json(
    output_dir_exp4 / "exp4-train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
val_4.to_json(
    output_dir_exp4 / "exp4-val.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
test_4.to_json(
    output_dir_exp4 / "exp4-eval.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

In [ ]:
print_stats(train_3, "Experience 3 Train", df_old)
print_stats(val_3, "Experience 3 Validation", df_old)
print_stats(test_3, "Experience 3 Test", df_old)

In [ ]:
# check if all models in test are in train
def check_models_in_train(train_df, test_df, experience_name):
    train_models = set(train_df["model_name"].unique())
    test_models = set(test_df["model_name"].unique())
    missing_models = test_models - train_models
    if len(missing_models) == 0:
        print(f"All models in test are present in train for {experience_name}.")
    else:
        print(f"Missing models in train for {experience_name}: {missing_models}")


check_models_in_train(train_3, test_3, "Experience 3")
check_models_in_train(train_4, test_4, "Experience 4")

check_models_in_train(train_3, val_3, "Experience 3")
check_models_in_train(train_4, val_4, "Experience 4")